## 🎯 Learning Objectives
* Understand the necessity of statefulness in advanced AI agents for long-running and complex tasks.
* Learn how to implement stateful agents using LangChain's built-in message history and checkpointing mechanisms.
* Be able to save and restore an agent's execution state to ensure resilience and enable advanced workflows.
* Analyze the performance implications and identify appropriate use cases for stateful agents with checkpointing.


## Stateful Agents with Checkpointing: Building Resilient and Persistent AI Workflows

In the realm of advanced AI agents, the ability to remember past interactions, decisions, and intermediate results is not just a feature—it's a fundamental requirement for building truly intelligent and robust systems. This is where **stateful agents** come into play. Unlike stateless agents, which treat every interaction as a fresh start, stateful agents maintain a memory of their ongoing conversation or task execution.

Imagine an AI agent designed to help you plan a complex multi-day trip. If it's stateless, every time you ask a follow-up question (e.g., "What about hotels in Paris?"), it would forget your previous preferences for flight times, budget, or destination cities. A stateful agent, however, retains this context, allowing for a natural, coherent, and efficient interaction over extended periods.

### Why Statefulness Matters in 2026

As agentic systems move from experimental prototypes to production-grade solutions, they encounter real-world challenges:

1.  **Long-running tasks**: Many complex tasks, like project management, scientific research, or customer onboarding, span hours, days, or even weeks. An agent needs to remember its progress.
2.  **Human-in-the-loop workflows**: Agents often collaborate with humans. If a human needs to step away and return, the agent must resume exactly where it left off.
3.  **Resilience and fault tolerance**: Production systems can crash, networks can fail, or LLM APIs might temporarily be unavailable. Losing all progress means starting over, which is unacceptable for critical applications.
4.  **Complex decision-making**: Agents exploring multiple paths or performing iterative refinement need to track their current state, previous attempts, and learned information.

### Introducing Checkpointing: The Save Game for Your Agent

**Checkpointing** is the mechanism that enables this persistence. Think of it like saving your progress in a video game. When you save, the game's entire state (your character's location, inventory, quest progress, etc.) is recorded. If the game crashes or you quit and return later, you can load that save file and pick up exactly where you left off.

For AI agents, checkpointing means serializing and storing the agent's internal state—its message history, tool outputs, internal thoughts, and any other relevant data—at specific points during its execution. This allows the agent to:

*   **Resume from failure**: If an agent process terminates unexpectedly, it can be restarted and restored to its last known good state, minimizing lost work.
*   **Pause and resume**: Users or other systems can explicitly pause an agent's execution and resume it later, even across different sessions or machines.
*   **Audit and debug**: Checkpoints provide snapshots of the agent's state at various stages, invaluable for understanding its behavior, debugging issues, or analyzing decision paths.
*   **A/B testing and experimentation**: Different agent configurations or prompts can be tested from a common starting state, ensuring fair comparison.

LangChain, a leading framework for building LLM-powered applications, provides robust abstractions for managing state and implementing checkpointing, making it straightforward to build resilient and persistent agent systems ready for the demands of 2026 and beyond. We'll leverage `RunnableWithMessageHistory` and `checkpointers` to achieve this.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install langchain langchain-openai

import os
from typing import List, Tuple, Dict, Any

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables import Runnable, RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# For checkpointing, we'll use a simple in-memory saver for demonstration.
# In production, you'd use a persistent store like SQLSaver, RedisSaver, etc.
from langchain.memory import ChatMessageHistory
from langchain.schema.runnable.config import RunnableConfig
from langchain.schema.runnable import RunnableLambda

# --- Configuration --- 
# Set your OpenAI API key. For production, use environment variables or a secure secret management system.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Using a mock LLM for reproducibility if you don't have an API key, 
# otherwise uncomment the ChatOpenAI line.
# from langchain_core.language_models import BaseChatModel
# class MockChatModel(BaseChatModel):
#     def invoke(self, messages: List[BaseMessage], **kwargs) -> BaseMessage:
#         last_message = messages[-1].content
#         if "hello" in last_message.lower():
#             return AIMessage(content="Hello there! How can I help you today?")
#         elif "remember" in last_message.lower():
#             return AIMessage(content="Yes, I remember our conversation. What would you like to recall?")
#         else:
#             return AIMessage(content=f"You said: '{last_message}'. I am a mock LLM.")
#     @property
#     def _llm_type(self) -> str:
#         return "mock_chat_model"

# llm = MockChatModel()

# For a real LLM, uncomment this and provide your API key
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- 1. Define the core agent logic --- 
# This is a simple conversational chain. In a real agent, this would involve tools, planning, etc.

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant. Keep your responses concise and relevant to the conversation history."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

# Our core chain: prompt -> LLM -> output parser
# This chain doesn't inherently know about history; it expects history to be passed in.
core_agent_chain = prompt | llm | StrOutputParser()

# --- 2. Implement a custom message history store for checkpointing --- 
# In a real application, this would be backed by a database (e.g., Redis, PostgreSQL).
# For this example, we'll simulate a persistent store using a dictionary.

# This dictionary will act as our "database" for session histories.
# Key: session_id, Value: ChatMessageHistory object
store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# --- 3. Wrap the agent with RunnableWithMessageHistory for statefulness and checkpointing --- 
# This runnable automatically manages message history and can be configured for checkpointing.

# The `RunnableWithMessageHistory` takes our core chain and a function to retrieve history.
# It also handles adding new messages to the history after each interaction.
conversational_agent = RunnableWithMessageHistory(
    core_agent_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# --- 4. Demonstrate Checkpointing --- 

# Define a session ID for our agent's conversation.
# In a real app, this would come from a user session, a task ID, etc.
SESSION_ID = "user_session_123"

# Configuration for the runnable, including the session ID.
# This is how LangChain knows which history to load/save.
config = {"configurable": {"session_id": SESSION_ID}}

print(f"--- Starting a new conversation for session: {SESSION_ID} ---")

# First interaction: The agent has no prior history.
response1 = conversational_agent.invoke(
    {"input": "Hi there! My name is Alice."}, 
    config=config
)
print(f"Alice: Hi there! My name is Alice.")
print(f"Agent: {response1}")

# Second interaction: The agent should remember Alice's name.
response2 = conversational_agent.invoke(
    {"input": "Can you tell me a fun fact about Python programming?"}, 
    config=config
)
print(f"Alice: Can you tell me a fun fact about Python programming?")
print(f"Agent: {response2}")

print(f"\n--- Simulating a 'crash' or 'pause' and then resuming ---")

# Clear the in-memory store to simulate a crash or restart of the application.
# In a real scenario, the 'store' would persist across application restarts.
# For this example, we're just showing that the *concept* of the store is external.
# If using SQLSaver, you wouldn't clear it; the data would remain in the DB.
# For this specific `get_session_history` implementation, clearing `store` effectively resets it.
# However, the `RunnableWithMessageHistory` itself will still interact with `get_session_history`.
# The key is that `get_session_history` *re-creates* the history if not found, 
# simulating a fresh start if the underlying persistent store is empty.

# To truly demonstrate checkpointing, we need to show that the history *persists* 
# even if the `ChatMessageHistory` object itself is gone from memory.
# Let's re-initialize the `store` to simulate a fresh application start.
# In a real app, the `get_session_history` would load from a persistent DB.

# Let's modify `get_session_history` to be more explicit about persistence.
# For this example, we'll just show the history *before* and *after* a simulated restart.

print(f"\n--- Current history in store for {SESSION_ID} before 'restart': ---")
current_history = get_session_history(SESSION_ID)
for msg in current_history.messages:
    print(f"  {msg.type.capitalize()}: {msg.content}")

# Simulate a complete application restart by clearing the in-memory store.
# In a real scenario, this 'store' would be a database that survives restarts.
# For this demonstration, we'll just show that the agent *can* retrieve history
# if the `get_session_history` function is correctly implemented to load from a persistent source.
# Since our `store` is global, we'll just pretend the application restarted and the `store` is still there
# but the *agent instance* is new.

# Let's create a *new* conversational agent instance to simulate a restart.
# The `get_session_history` function will ensure the history is loaded from `store`.
new_conversational_agent = RunnableWithMessageHistory(
    core_agent_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print(f"\n--- Resuming conversation with a 'new' agent instance for session: {SESSION_ID} ---")

# Third interaction: The new agent instance should still remember Alice's name and the previous topic.
response3 = new_conversational_agent.invoke(
    {"input": "So, what's your favorite programming language then?"}, 
    config=config
)
print(f"Alice: So, what's your favorite programming language then?")
print(f"Agent: {response3}")

print(f"\n--- Final history in store for {SESSION_ID} after all interactions: ---")
final_history = get_session_history(SESSION_ID)
for msg in final_history.messages:
    print(f"  {msg.type.capitalize()}: {msg.content}")

# --- Advanced Checkpointing (using LangChain's checkpointers) ---
# For more robust checkpointing, LangChain provides `BaseCheckpointSaver` implementations.
# Let's demonstrate with a simple `MemorySaver`.

from langchain.memory import ConversationBufferMemory
from langchain.schema.runnable import RunnableConfig
from langchain.schema.runnable.utils import ConfigurableFieldSpec
from langchain.schema.runnable import RunnableLambda

# A more explicit way to manage state with a checkpointer.
# This is often used with LangGraph or more complex agentic workflows.

# Define a state schema (simple dictionary for this example)
class AgentState(dict):
    messages: List[BaseMessage]

# A simple checkpointer (in-memory for demo, replace with SQLSaver/RedisSaver for production)
from langchain.memory import BaseCheckpointSaver, MemorySaver

# Initialize the MemorySaver. This will store checkpoints in memory.
# In a real application, this would be configured to use a persistent database.
checkpoint_saver = MemorySaver()

# Let's create a slightly different chain that explicitly uses the checkpoint_saver.
# This is more common in LangGraph, but we can illustrate the concept here.

# A simple chain that just echoes the input and adds it to history.
# In a real agent, this would be your complex agent logic.

def process_input_with_history(state: AgentState, input_message: HumanMessage) -> AgentState:
    # Simulate agent processing
    ai_response_content = f"Acknowledged: {input_message.content}"
    ai_message = AIMessage(content=ai_response_content)
    new_messages = state.get("messages", []) + [input_message, ai_message]
    return {"messages": new_messages}

# This runnable will be the core logic that updates the state.
# We'll wrap it with a `RunnableLambda` to handle the state transformation.
state_updating_runnable = RunnableLambda(process_input_with_history)

# The `with_config` method allows us to attach a checkpoint saver.
# This is how you tell LangChain to use a specific saver for a runnable.
# The `configurable` field specifies how to identify the checkpoint (e.g., thread_id).

# Define a unique thread ID for this specific agent run.
THREAD_ID = "agent_thread_456"

# The runnable that will be checkpointed.
# For this example, we're just passing messages through and letting the saver handle it.
# In a real scenario, `state_updating_runnable` would be your agent's core logic.
checkpointed_chain = (RunnablePassthrough.assign(messages=lambda x: x["messages"] + [HumanMessage(content=x["input"])])
                      | RunnableLambda(lambda x: {"messages": x["messages"] + [AIMessage(content=f"Processed: {x['input']}")]}))

# This is how you make any runnable checkpointable.
# The `checkpoint_saver` will automatically save the state after each step.
# The `configurable` field defines the key for the checkpoint (e.g., thread_id).
checkpointed_runnable = checkpointed_chain.with_config(
    configurable={
        "thread_id": THREAD_ID,
        "checkpoint_saver": checkpoint_saver # Explicitly pass the saver
    }
)

print(f"\n--- Demonstrating explicit checkpointing with MemorySaver for thread: {THREAD_ID} ---")

# Initial state (empty messages)
initial_state = {"messages": []}

# First interaction
print(f"\n--- First interaction (thread {THREAD_ID}) ---")
output1 = checkpointed_runnable.invoke({"input": "Start task A", "messages": initial_state["messages"]})
print(f"Output 1: {output1}")

# The state is automatically saved by `checkpoint_saver`.
# We can retrieve it directly from the saver.
retrieved_checkpoint_1 = checkpoint_saver.get_checkpoint({"configurable": {"thread_id": THREAD_ID}})
print(f"Retrieved checkpoint 1: {retrieved_checkpoint_1}")

# Simulate a restart: Load the last checkpoint and continue.
print(f"\n--- Simulating restart and second interaction (thread {THREAD_ID}) ---")

# To resume, we need to provide the last known state as input to the runnable.
# The `checkpoint_saver` will load the state associated with `THREAD_ID`.
# The `RunnableWithMessageHistory` handles this more automatically, but for raw checkpointers,
# you often explicitly load the state.

# For this example, we'll manually load the state and pass it.
# In a more integrated system (like LangGraph), this is handled more seamlessly.

# Let's use the `get_state` method of the `checkpoint_saver` to retrieve the full state.
# Note: `get_state` is a conceptual method; `MemorySaver` stores directly.
# For `MemorySaver`, the checkpoint *is* the state.

# The `checkpoint_saver` stores the *entire* state of the runnable, not just messages.
# For this simple example, the state is just the messages.

# Let's refine the checkpointed_runnable to be more explicit about state.
# This is closer to how LangGraph manages state.

# Define a simple graph-like structure for state updates
from typing import TypedDict

class AgentGraphState(TypedDict):
    chat_history: List[BaseMessage]
    user_input: str

# A simple node that processes input and updates history
def process_step(state: AgentGraphState) -> AgentGraphState:
    user_message = HumanMessage(content=state["user_input"])
    ai_response = AIMessage(content=f"Processed: {state['user_input']}")
    new_history = state["chat_history"] + [user_message, ai_response]
    return {"chat_history": new_history, "user_input": ""} # Clear user_input after processing

# Create a runnable from this function
process_step_runnable = RunnableLambda(process_step)

# Make it checkpointable
checkpointed_graph_runnable = process_step_runnable.with_config(
    configurable={
        "thread_id": THREAD_ID,
        "checkpoint_saver": checkpoint_saver
    }
)

print(f"\n--- Demonstrating explicit checkpointing with a stateful runnable (LangGraph-like) ---")

# Initial state for the graph-like runnable
initial_graph_state: AgentGraphState = {"chat_history": [], "user_input": ""}

# First interaction
print(f"\n--- First interaction (graph thread {THREAD_ID}) ---")
output_graph_1 = checkpointed_graph_runnable.invoke(
    {"user_input": "Hello, I need to order supplies.", "chat_history": initial_graph_state["chat_history"]}
)
print(f"Output 1 (graph): {output_graph_1}")

# The state is saved. Let's retrieve it.
retrieved_graph_checkpoint_1 = checkpoint_saver.get_checkpoint({"configurable": {"thread_id": THREAD_ID}})
print(f"Retrieved graph checkpoint 1: {retrieved_graph_checkpoint_1}")

# Simulate restart and resume
print(f"\n--- Simulating restart and second interaction (graph thread {THREAD_ID}) ---")

# To resume, we need to load the last state and pass it to the next invoke.
# The `get_checkpoint` method returns the full state.
loaded_state = retrieved_graph_checkpoint_1["channel_values"]

# Invoke with the loaded history and new input
output_graph_2 = checkpointed_graph_runnable.invoke(
    {"user_input": "I need 5 units of item X and 10 units of item Y.", 
     "chat_history": loaded_state["chat_history"]}
)
print(f"Output 2 (graph): {output_graph_2}")

# Verify the final state in the checkpoint saver
final_graph_checkpoint = checkpoint_saver.get_checkpoint({"configurable": {"thread_id": THREAD_ID}})
print(f"\nFinal graph checkpoint after all interactions: {final_graph_checkpoint}")

# You can see the chat_history now contains all messages from both interactions.


### Interpreting the Code Output and Performance Considerations

In the first part of the code, using `RunnableWithMessageHistory`, you'll observe that the agent remembers previous turns of the conversation. Even after we conceptually "restart" the application (by creating a new `conversational_agent` instance), the agent is able to retrieve the full conversation history from our `store` (which simulates a persistent database). This demonstrates how `RunnableWithMessageHistory` automatically manages the `ChatMessageHistory` for a given `session_id`, making the agent stateful.

The second part, using `MemorySaver` with a more explicit state management (`AgentGraphState`), illustrates how you can checkpoint the *entire state* of a more complex agent or graph. After the first interaction, we retrieve the checkpoint, which contains the updated `chat_history`. When we invoke the runnable again with new input, we explicitly pass the `chat_history` loaded from the previous checkpoint, demonstrating how an agent can resume from a saved state.

#### Performance Trade-offs and Use Cases:

1.  **Serialization/Deserialization Overhead**: Saving and loading state involves converting Python objects into a storable format (serialization) and back (deserialization). For very large states (e.g., complex internal knowledge bases, extensive tool outputs), this can introduce latency. Choose efficient serialization formats (e.g., JSON, Protocol Buffers) and optimize your state structure.
2.  **Storage Backend Choice**: The choice of checkpointing backend is critical:
    *   **In-memory (e.g., `MemorySaver`)**: Fastest for development and testing, but loses state on application restart. Not suitable for production.
    *   **Database (e.g., `SQLSaver`, Redis, MongoDB)**: Provides persistence, scalability, and often transactional guarantees. Redis is excellent for high-throughput, low-latency state storage. PostgreSQL or other relational databases are good for structured state and complex queries.
    *   **Cloud Storage (e.g., S3, GCS)**: Suitable for very large, less frequently accessed states, or for archival purposes. Can be slower due to network latency.
3.  **Checkpoint Frequency**: How often should you save a checkpoint? Too often, and you incur high overhead. Too rarely, and you risk losing significant progress on failure. This is a design decision based on the task's criticality and the acceptable data loss window.
4.  **State Size**: Keep the agent's state as lean as possible. Only store what's absolutely necessary for resuming or decision-making. Large states increase storage costs, I/O latency, and serialization overhead.

#### Typical Use Cases:

*   **Customer Service Bots**: Remembering user preferences, past issues, and ongoing ticket status across multiple interactions or even days.
*   **Long-running Data Analysis Agents**: An agent performing complex data cleaning, feature engineering, or model training can save its progress, allowing it to resume if interrupted or to be reviewed by a human at intermediate steps.
*   **Interactive Content Generation**: Agents assisting with writing, design, or coding can maintain context of the project, user feedback, and generated drafts.
*   **Complex Planning and Orchestration**: Agents managing multi-step processes (e.g., supply chain optimization, software deployment) can checkpoint their current plan, executed steps, and remaining tasks.
*   **Human-in-the-Loop AI**: When human review or intervention is required, checkpointing allows the agent to pause, present its current state to a human, and then resume based on human feedback, ensuring seamless collaboration.

By carefully considering these factors, developers can build highly resilient, efficient, and user-friendly AI agents that can tackle complex, real-world problems in 2026 and beyond.


### Resources

*   **LangChain Documentation on Message History**: [https://python.langchain.com/docs/modules/memory/chat_messages/](https://python.langchain.com/docs/modules/memory/chat_messages/)
*   **LangChain Documentation on Checkpointers (for LangGraph)**: [https://python.langchain.com/docs/langgraph/how-to/checkpoints](https://python.langchain.com/docs/langgraph/how-to/checkpoints)
*   **LangChain Expression Language (LCEL) Guide**: [https://python.langchain.com/docs/expression_language/](https://python.langchain.com/docs/expression_language/)
*   **LangGraph Documentation (for advanced state management)**: [https://python.langchain.com/docs/langgraph](https://python.langchain.com/docs/langgraph)
*   **Redis for Persistent Storage**: [https://redis.io/](https://redis.io/)
*   **PostgreSQL for Structured Data Storage**: [https://www.postgresql.org/](https://www.postgresql.org/)
